# ForgeX — Model Training & Evaluation

This notebook demonstrates a multi-output Random Forest surrogate for ForgeX material quantity and material-cost estimates. The source records are catalog/template reference estimates, not measured production data; ForgeX's deterministic calculations remain the source of truth.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
assert DATA_DIR.is_dir(), f"Could not locate project data directory from {Path.cwd()}"
print(f"Project root: {PROJECT_ROOT}")

## 1. Source data and training examples

The repository contains no CSV training set or model-training script. This notebook uses the actual `data/materials.json` catalog and design templates in `data/design_templates/`. ForgeX's existing calculation equations are MAS-01 (`mass = volume × density`) and CST-01 (`material cost = mass × cost per kg`). The labeled examples below are reproducibly derived by applying those equations to every volume-bearing template and printable material in the catalog; they are not independent experimental observations.

In [ ]:
materials_path = DATA_DIR / "materials.json"
with materials_path.open(encoding="utf-8") as file:
    materials_catalog = json.load(file)

def finite_number(value):
    return isinstance(value, (int, float)) and np.isfinite(value)

print_materials = [
    record
    for record in materials_catalog["records"]
    if "FDM-printable" in record.get("printability_notes", "")
    and finite_number(record.get("density_kg_m3"))
    and finite_number(record.get("cost_per_kg_usd"))
]

template_records = []
for template_path in sorted((DATA_DIR / "design_templates").glob("*.json")):
    with template_path.open(encoding="utf-8") as file:
        template = json.load(file)
    parameters = template.get("parameters", {})
    volume_cm3 = parameters.get("estimated_volume_cm3")
    if finite_number(volume_cm3) and volume_cm3 > 0:
        template_records.append((template, float(volume_cm3)))

rows = []
for template, volume_cm3 in template_records:
    for material in print_materials:
        density_kg_m3 = float(material["density_kg_m3"])
        cost_per_kg_usd = float(material["cost_per_kg_usd"])
        estimated_mass_kg = volume_cm3 * 1e-6 * density_kg_m3
        estimated_cost_usd = estimated_mass_kg * cost_per_kg_usd
        rows.append({
            "template_id": template["template_id"],
            "template_name": template["name"],
            "material": material["name"],
            "estimated_volume_cm3": volume_cm3,
            "density_kg_m3": density_kg_m3,
            "cost_per_kg_usd": cost_per_kg_usd,
            "estimated_mass_kg": estimated_mass_kg,
            "estimated_cost_usd": estimated_cost_usd,
        })

df = pd.DataFrame(rows)
assert not df.empty, "No source material/template combinations were found."
assert {"estimated_mass_kg", "estimated_cost_usd"}.issubset(df.columns)
print(f"Source: {materials_path.relative_to(PROJECT_ROOT)} + {len(template_records)} volume-bearing templates")
print(f"Derived examples: {len(df)} ({len(template_records)} templates × {len(print_materials)} printable materials)")
print("Target labels are calculated ForgeX reference estimates, not measured data.")

## 2. Explore the derived dataset

Inspect the actual rows, types, and ranges before fitting. Because this is a small catalog-derived demonstration dataset, the resulting scores should not be interpreted as production generalization performance.

In [ ]:
display(df.head())
df.info()
display(df.describe(include="all"))

## 3. Feature/target preparation and 80/20 split

The numeric inputs are the real ForgeX calculation inputs: nominal solid volume, catalog density, and catalog material price. The model predicts both existing calculation outputs with one multi-output regressor. Template and material names are retained for traceability but excluded from the numeric training features.

In [ ]:
feature_columns = ["estimated_volume_cm3", "density_kg_m3", "cost_per_kg_usd"]
target_columns = ["estimated_mass_kg", "estimated_cost_usd"]
X = df[feature_columns]
y = df[target_columns]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Features: {feature_columns}")
print(f"Targets: {target_columns}")
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)} | Total: {len(df)}")

## 4. Train the Random Forest

`RandomForestRegressor` supports multiple numeric targets directly. This fitted estimator is a demonstration surrogate only: the labeled values come from ForgeX's closed-form equations, and the deterministic calculator remains authoritative.

In [ ]:
model = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Random Forest fitted for targets:", target_columns)

## 5. Evaluate on the held-out test split

R² and MAE are reported separately for each target and as a simple mean across targets. With only a few derived rows and no measured labels, these metrics demonstrate the workflow, not validated predictive accuracy.

In [ ]:
y_pred_df = pd.DataFrame(y_pred, columns=target_columns, index=y_test.index)
eval_results = {}
for target in target_columns:
    actual = y_test[target].to_numpy()
    predicted = y_pred_df[target].to_numpy()
    target_r2 = r2_score(actual, predicted)
    target_mae = mean_absolute_error(actual, predicted)
    eval_results[target] = {"r2": float(target_r2), "mae": float(target_mae)}
    print(f"{target}: R² = {target_r2:.4f} | MAE = {target_mae:.6f}")

mean_r2 = float(np.mean([metrics["r2"] for metrics in eval_results.values()]))
mean_mae = float(np.mean([metrics["mae"] for metrics in eval_results.values()]))
print(f"\nMean across targets: R² = {mean_r2:.4f} | MAE = {mean_mae:.6f}")

## 6. Feature importance

The estimator exposes one importance vector for this multi-output fit. Use it as a qualitative view of this small, derived demonstration set—not as evidence that a feature is causally important in real manufacturing.

In [ ]:
feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False, ignore_index=True)
display(feature_importance)

## 7. Export the trained model

Save the fitted estimator at the project root alongside the declared feature/target schema and evaluation metrics so it can be reloaded reproducibly. It is not wired into the ForgeX application.

In [ ]:
model_artifact = {
    "model": model,
    "feature_columns": feature_columns,
    "target_columns": target_columns,
    "evaluation": eval_results,
    "data_provenance": {
        "materials_catalog": str(materials_path.relative_to(PROJECT_ROOT)),
        "template_directory": "data/design_templates/",
        "label_method": "Derived using ForgeX MAS-01 and CST-01 equations; not measured observations.",
    },
}
model_path = PROJECT_ROOT / "forgex_model.joblib"
joblib.dump(model_artifact, model_path)
print(f"Exported model artifact: {model_path}")
print(f"Targets: {target_columns}")